# SRQ-FLY D3 — CUB train-only replication

Run every cell in order on a Colab T4 GPU. Hyperparameters are selected on an inner split of CUB training data; outer validation is evaluated once. This notebook never extracts or evaluates CUB test features.

In [ ]:
# === Edit path/source values only. Do not edit protocol values. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'feature/srq-fly-d3-cub-replication'
WORK_DIR = '/content/SOHO-CL'
DRIVE_ROOT = '/content/drive/MyDrive/T-SOHO'
DRIVE_TRAIN_CACHE = f'{DRIVE_ROOT}/cub_train_feature_cache'
TRAIN_CACHE_DIR = '/content/cub_train_feature_cache'
DRIVE_LARGE_WTA = f'{DRIVE_ROOT}/srq_fly_cub_wta_h10000_seed2025'
LARGE_WTA_DIR = '/content/srq_fly_cub_wta_h10000_seed2025'
DRIVE_MATCHED_WTA = f'{DRIVE_ROOT}/srq_fly_cub_wta_h4518_seed2025'
MATCHED_WTA_DIR = '/content/srq_fly_cub_wta_h4518_seed2025'
OUTPUT_DIR = f'{DRIVE_ROOT}/srq_fly_cub_d3_train_only_seed2025'
CHECKPOINT_SOURCE = 'huggingface'  # or 'google_drive'
DRIVE_CHECKPOINT_PATH = f'{DRIVE_ROOT}/model.safetensors'
BATCH_SIZE = 128
CHECKPOINT_SIZE = 346284714
CHECKPOINT_SHA256 = '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CONFIG_SHA256 = '26926c625aa6dbebf7271a4767d73a56e9bfe3bf0e139edca012977789dec772'


In [ ]:
# Runtime setup. chdir before replacing the repository.
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
os.chdir('/content')
repo = Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
clone = subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_GIT_URL, WORK_DIR], text=True, capture_output=True)
print(clone.stdout, clone.stderr, sep='')
assert clone.returncode == 0, 'Clone failed. Confirm the D3 branch was pushed.'
os.chdir(WORK_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt', 'kagglehub', 'huggingface_hub'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
config_path = Path('configs/srq_fly_cub_d3_train_only.json')
assert hashlib.sha256(config_path.read_bytes()).hexdigest() == CONFIG_SHA256
print('repo commit:', commit)
print('GPU:', torch.cuda.get_device_name(0))
print('locked config:', CONFIG_SHA256)


In [ ]:
# Obtain and verify the exact frozen ViT checkpoint.
if CHECKPOINT_SOURCE == 'huggingface':
    from huggingface_hub import hf_hub_download
    CHECKPOINT_PATH = hf_hub_download('timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', 'model.safetensors')
elif CHECKPOINT_SOURCE == 'google_drive':
    CHECKPOINT_PATH = DRIVE_CHECKPOINT_PATH
else:
    raise ValueError('CHECKPOINT_SOURCE must be huggingface or google_drive')
checkpoint = Path(CHECKPOINT_PATH)
digest = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
assert checkpoint.stat().st_size == CHECKPOINT_SIZE
assert digest == CHECKPOINT_SHA256
print('checkpoint PASS:', checkpoint, digest)


In [ ]:
# Download and identity-audit the processed CUB artifact. Audit does not run the backbone.
import kagglehub
DATASET_PATH = kagglehub.dataset_download('zaphat206/cub-200-2011')
DATASET_AUDIT = '/content/cub_dataset_audit_d3.json'
audit_command = [sys.executable, '-u', 'tools/cub_dataset_audit.py', '--root', DATASET_PATH, '--output', DATASET_AUDIT, '--expected-identity-sha256', 'e374af9b576cb6b3503198ef3ea30fd0aa9d2e18c230ff8064e21d4f644af2ca', '--progress-every', '1000']
subprocess.run(audit_command, check=True)
audit = json.loads(Path(DATASET_AUDIT).read_text())
print('CUB identity PASS:', audit['dataset_identity_sha256'])
print('train/test inventory:', audit['train']['image_count'], audit['test']['image_count'])


In [ ]:
# D3 correctness gate. Synthetic only; no CUB test feature is opened.
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_srq_fly_d3_cub.py', 'tests/test_srq_fly_learner.py', 'tests/test_srq_fly_math.py', 'tests/test_srq_fly_d1.py'], check=True)
print('SRQ-FLY D3 correctness gate: PASS')


In [ ]:
# Restore or extract TRAIN embeddings only, with bounded copy/extraction progress.
def copy_file_progress(source, target, chunk=32*2**20):
    source, target = Path(source), Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file() and target.stat().st_size == source.stat().st_size:
        print('RESTORED', target.name, f'{target.stat().st_size/2**20:.1f} MiB')
        return
    partial = target.with_suffix(target.suffix + '.partial')
    partial.unlink(missing_ok=True)
    copied, total, next_report = 0, source.stat().st_size, 10
    with source.open('rb') as reader, partial.open('wb') as writer:
        while block := reader.read(chunk):
            writer.write(block); copied += len(block)
            percent = int(100 * copied / total)
            if percent >= next_report:
                print(f'COPY {target.name}: {percent}% ({copied/2**20:.1f}/{total/2**20:.1f} MiB)', flush=True)
                next_report += 10
    partial.replace(target)

local_cache, drive_cache = Path(TRAIN_CACHE_DIR), Path(DRIVE_TRAIN_CACHE)
if not (local_cache / 'metadata.json').is_file():
    if (drive_cache / 'metadata.json').is_file():
        copy_file_progress(drive_cache / 'train.pt', local_cache / 'train.pt')
        copy_file_progress(drive_cache / 'metadata.json', local_cache / 'metadata.json')
        print('Restored train-only CUB feature cache from Drive.')
    else:
        command = [sys.executable, '-u', 'tools/experiment_runner.py', '--extract-features-only', '--extract-train-only', '--root', DATASET_PATH, '--backbone-checkpoint', CHECKPOINT_PATH, '--backbone-checkpoint-size', str(CHECKPOINT_SIZE), '--backbone-checkpoint-sha256', CHECKPOINT_SHA256, '--feature-cache-dir', TRAIN_CACHE_DIR, '--output-dir', '/content/d3_extract_only', '--dataset', 'CUB-200-2011', '--model-name', 'vit_base_patch16_224', '--data-augmentation', 'vit', '--seed', '2025', '--num-classes', '200', '--num-tasks', '20', '--device', 'cuda', '--batch-size', str(BATCH_SIZE), '--num-workers', '2']
        print('EXTRACT START: one progress line per train task.', flush=True)
        subprocess.run(command, check=True)
        assert not (local_cache / 'test.pt').exists()
        if drive_cache.exists(): raise RuntimeError('Incomplete Drive cache exists; inspect it instead of overwriting.')
        copy_file_progress(local_cache / 'train.pt', drive_cache / 'train.pt')
        copy_file_progress(local_cache / 'metadata.json', drive_cache / 'metadata.json')
metadata = json.loads((local_cache / 'metadata.json').read_text())
assert metadata['test_features_materialized'] is False
assert not (local_cache / 'test.pt').exists()
print('train cache PASS:', metadata['train_shape'], '| test.pt absent')


In [ ]:
# Restore WTA infrastructure when available; then run locked D3 with live progress.
def restore_directory(source, target):
    source, target = Path(source), Path(target)
    if not (source / 'metadata.json').is_file(): return False
    for name in ('projection.pt', 'train_codes.pt', 'metadata.json'):
        copy_file_progress(source / name, target / name)
    return True

restore_directory(DRIVE_LARGE_WTA, LARGE_WTA_DIR)
restore_directory(DRIVE_MATCHED_WTA, MATCHED_WTA_DIR)
output = Path(OUTPUT_DIR); output.mkdir(parents=True, exist_ok=True)
shutil.copy2(DATASET_AUDIT, output / 'cub_dataset_audit.json')
shutil.copy2('configs/srq_fly_cub_d3_train_only.json', output / 'locked_config.json')
(output / 'environment.json').write_text(json.dumps({'git_commit': commit, 'python': sys.version, 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0)}, indent=2))
command = [sys.executable, '-u', 'tools/srq_fly_d3_cub.py', '--config', 'configs/srq_fly_cub_d3_train_only.json', '--dataset-audit', DATASET_AUDIT, '--feature-cache-dir', TRAIN_CACHE_DIR, '--large-code-cache-dir', LARGE_WTA_DIR, '--matched-code-cache-dir', MATCHED_WTA_DIR, '--output-dir', OUTPUT_DIR, '--device', 'cuda', '--require-test-hidden']
print('Starting/resuming D3. Watch CACHE, INNER START/DONE, LOCKED, OUTER, and TASK lines.', flush=True)
log_path = output / 'd3_run.log'
with log_path.open('a', encoding='utf-8') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True); log.write(line); log.flush()
    returncode = process.wait()
for local, drive_path in ((LARGE_WTA_DIR, DRIVE_LARGE_WTA), (MATCHED_WTA_DIR, DRIVE_MATCHED_WTA)):
    local, drive_path = Path(local), Path(drive_path)
    if (local / 'metadata.json').is_file():
        for name in ('projection.pt', 'train_codes.pt', 'metadata.json'):
            copy_file_progress(local / name, drive_path / name)
assert returncode == 0, 'D3 runner failed; return the complete traceback without editing config.'
assert (output / 'd3_results.json').is_file()
print('D3 process COMPLETE')


In [ ]:
# Display the train-only result and download evidence. STOP after this cell.
import pandas as pd
result = json.loads((Path(OUTPUT_DIR) / 'd3_results.json').read_text())
rows = []
for item in result['results']:
    rows.append({'method': item['method'], 'lambda': item['ridge_lambda'], 'outer_AA': item['validation_average_accuracy'], 'outer_final': item['stage_accuracy'][-1], 'state_bytes': item['persistent_state_bytes'], 'max_residual': item['maximum_solver_relative_residual']})
display(pd.DataFrame(rows).sort_values('outer_AA', ascending=False))
print('status:', result['status'])
print('comparison:', json.dumps(result['comparison'], indent=2))
print('gates:', json.dumps(result['gates'], indent=2))
archive = shutil.make_archive('/content/srq_fly_cub_d3_train_only', 'zip', root_dir=OUTPUT_DIR)
print('artifact SHA-256:', hashlib.sha256(Path(archive).read_bytes()).hexdigest())
from google.colab import files
files.download(archive)
print('STOP. Return the ZIP for audit; do not evaluate CUB test.')
